In [1]:
import pandas as pd

from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [2]:
def ler_sinasc(arquivo):
    df = pd.read_csv(
        arquivo,
        sep=';',
        skiprows=3,
        encoding='latin1'
    )

    df = df.dropna(how='all')

    return df

In [3]:
nasc_estado = ler_sinasc('quedas nascimento por estado.csv')
idade_mae = ler_sinasc('idade da mãe.csv')
instrucao_mae = ler_sinasc('instrução da mãe.csv')
pre_natal = ler_sinasc('consultas pre-natal.csv')
tipo_parto = ler_sinasc('tipo de parto.csv')

print(nasc_estado.head())
print(idade_mae.head())
print(instrucao_mae.head())
print(pre_natal.head())
print(tipo_parto.head())

  Região/Unidade da Federação      2010      2011      2012      2013  \
0                Região Norte  306422.0  313745.0  308375.0  313272.0   
1                 .. Rondônia   25835.0   27658.0   26513.0   27097.0   
2                     .. Acre   16495.0   17817.0   16700.0   17075.0   
3                 .. Amazonas   74188.0   76202.0   77434.0   79041.0   
4                  .. Roraima    9738.0    9945.0   10601.0   10814.0   

       2014      2015      2016      2017      2018      2019      2020  \
0  321682.0  320924.0  307526.0  312682.0  319228.0  313696.0  301635.0   
1   27560.0   27918.0   26602.0   27503.0   28091.0   27028.0   25798.0   
2   17139.0   16980.0   15773.0   16358.0   16543.0   16280.0   15142.0   
3   81145.0   80097.0   76703.0   78066.0   78087.0   77622.0   75635.0   
4   11120.0   11412.0   11376.0   11737.0   13344.0   14620.0   13760.0   

       2021      2022      2023      2024      Total  
0  309362.0  289158.0  284197.0  265670.0  4587574.0  


In [4]:
anos_brasil = [str(ano) for ano in range(2010, 2025)]

In [5]:
def transformar_longo(df, nome_coluna):
    coluna = df.columns[0]

    df = df.dropna(how='all').copy()

    df_long = df.melt(
        id_vars=[coluna],
        value_vars=anos_brasil,
        var_name='Ano',
        value_name='Valor'
    )

    df_long = df_long.rename(columns={coluna: nome_coluna})

    df_long['Valor'] = (
        df_long['Valor']
        .astype(str)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )

    df_long['Valor'] = pd.to_numeric(df_long['Valor'], errors='coerce')

    return df_long

In [6]:
idade_mae = idade_mae[
    idade_mae[idade_mae.columns[0]].isin([
        'Menor de 10 anos',
        '10 a 14 anos',
        '15 a 19 anos',
        '20 a 24 anos',
        '25 a 29 anos',
        '30 a 34 anos',
        '35 a 39 anos',
        '40 a 44 anos',
        '45 a 49 anos',
        '50 a 54 anos',
        '55 a 59 anos',
        '60 a 64 anos',
        '65 a 69 anos',
        'Idade ignorada',
        'Total'
    ])
]

instrucao_mae = instrucao_mae[
    instrucao_mae[instrucao_mae.columns[0]].isin([
        'Nenhuma',
        '1 a 3 anos',
        '4 a 7 anos',
        '8 a 11 anos',
        '12 anos e mais',
        'Ignorado',
        'Total'
    ])
]

pre_natal = pre_natal[
    pre_natal[pre_natal.columns[0]].isin([
        'Nenhuma',
        'De 1 a 3 consultas',
        'De 4 a 6 consultas',
        '7 ou mais consultas',
        'Ignorado',
        'Total'
    ])
]

tipo_parto = tipo_parto[
    tipo_parto[tipo_parto.columns[0]].isin([
        'Vaginal',
        'Cesário',
        'Ignorado',
        'Total'
    ])
]

In [7]:
idade_long = transformar_longo(idade_mae, 'Idade da mãe')
instrucao_long = transformar_longo(instrucao_mae, 'Instrução da mãe')
pre_natal_long = transformar_longo(pre_natal, 'Consultas pré-natal')
parto_long = transformar_longo(tipo_parto, 'Tipo de parto')

In [8]:
coluna_estado = nasc_estado.columns[0]

total_ano = nasc_estado[nasc_estado[coluna_estado] == 'Total']

total_ano = total_ano.melt(
    value_vars=anos_brasil,
    var_name='Ano',
    value_name='Nascimentos'
)

total_ano['Nascimentos'] = (
    total_ano['Nascimentos']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
)

total_ano['Nascimentos'] = pd.to_numeric(
    total_ano['Nascimentos'],
    errors='coerce'
)

total_ano['Nascimentos_anterior'] = total_ano['Nascimentos'].shift(1)

total_ano['Queda_nascimento'] = (
    total_ano['Nascimentos'] < total_ano['Nascimentos_anterior']
)

total_ano = total_ano.dropna()

print(total_ano)

     Ano  Nascimentos  Nascimentos_anterior  Queda_nascimento
1   2011     29131600            28618680.0             False
2   2012     29057890            29131600.0              True
3   2013     29040270            29057890.0              True
4   2014     29792590            29040270.0             False
5   2015     30176680            29792590.0             False
6   2016     28578000            30176680.0              True
7   2017     29235350            28578000.0             False
8   2018     29449320            29235350.0             False
9   2019     28491460            29449320.0              True
10  2020     27301450            28491460.0              True
11  2021     26771010            27301450.0              True
12  2022     25619220            26771010.0              True
13  2023     25375760            25619220.0              True
14  2024     23893250            25375760.0              True


In [9]:
def proporcao_categoria(df_long, coluna_categoria, categoria):
    base = df_long[
        df_long[coluna_categoria] == categoria
    ][['Ano', 'Valor']].copy()

    total = df_long.groupby('Ano')['Valor'].sum().reset_index()

    base = base.merge(
        total,
        on='Ano',
        suffixes=('_categoria', '_total')
    )

    base['Proporcao'] = base['Valor_categoria'] / base['Valor_total']

    return base[['Ano', 'Proporcao']]

In [10]:
maes_30_34 = proporcao_categoria(
    idade_long,
    'Idade da mãe',
    '30 a 34 anos'
)

instrucao_12 = proporcao_categoria(
    instrucao_long,
    'Instrução da mãe',
    '12 anos e mais'
)

pre_natal_7 = proporcao_categoria(
    pre_natal_long,
    'Consultas pré-natal',
    '7 ou mais consultas'
)

cesario = proporcao_categoria(
    parto_long,
    'Tipo de parto',
    'Cesário'
)

In [11]:
base_regras = total_ano[['Ano', 'Queda_nascimento']].copy()

base_regras = base_regras.merge(maes_30_34, on='Ano')
base_regras = base_regras.rename(columns={'Proporcao': 'Mae_30_34'})

base_regras = base_regras.merge(instrucao_12, on='Ano')
base_regras = base_regras.rename(columns={'Proporcao': 'Instrucao_12_mais'})

base_regras = base_regras.merge(pre_natal_7, on='Ano')
base_regras = base_regras.rename(columns={'Proporcao': 'Pre_natal_7_mais'})

base_regras = base_regras.merge(cesario, on='Ano')
base_regras = base_regras.rename(columns={'Proporcao': 'Cesario'})

print(base_regras)

     Ano  Queda_nascimento  Mae_30_34  Instrucao_12_mais  Pre_natal_7_mais  \
0   2011             False   0.090532           0.079371          0.306402   
1   2012              True   0.094256           0.075718          0.308458   
2   2013              True   0.096935           0.080773          0.312098   
3   2014             False   0.098174           0.086138          0.323088   
4   2015             False   0.099688           0.091679          0.332462   
5   2016              True   0.099743           0.094771          0.338725   
6   2017             False   0.101549           0.100289          0.346519   
7   2018             False   0.103859           0.104212          0.354228   
8   2019              True   0.104789           0.106373          0.362156   
9   2020              True   0.104265           0.108018          0.355095   
10  2021              True   0.102556           0.109351          0.365686   
11  2022              True   0.104498           0.115794        

In [12]:
base_binaria = pd.DataFrame()

base_binaria['Queda_nascimento'] = base_regras['Queda_nascimento']

base_binaria['Mae_30_34_alta'] = (
    base_regras['Mae_30_34'] > base_regras['Mae_30_34'].median()
)

base_binaria['Instrucao_12_mais_alta'] = (
    base_regras['Instrucao_12_mais'] > base_regras['Instrucao_12_mais'].median()
)

base_binaria['Pre_natal_7_mais_alto'] = (
    base_regras['Pre_natal_7_mais'] > base_regras['Pre_natal_7_mais'].median()
)

base_binaria['Cesario_alto'] = (
    base_regras['Cesario'] > base_regras['Cesario'].median()
)

print(base_binaria)

    Queda_nascimento  Mae_30_34_alta  Instrucao_12_mais_alta  \
0              False           False                   False   
1               True           False                   False   
2               True           False                   False   
3              False           False                   False   
4              False           False                   False   
5               True           False                   False   
6              False           False                   False   
7              False            True                    True   
8               True            True                    True   
9               True            True                    True   
10              True            True                    True   
11              True            True                    True   
12              True            True                    True   
13              True            True                    True   

    Pre_natal_7_mais_alto  Cesario_alto

In [14]:
itens_frequentes = apriori(
    base_binaria,
    min_support=0.3,
    use_colnames=True
)

regras = association_rules(
    itens_frequentes,
    metric='confidence',
    min_threshold=0.6
)

regras = regras.sort_values(
    by=['confidence', 'support'],
    ascending=False
)

print(regras[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

                 antecedents  \
8           (Mae_30_34_alta)   
9   (Instrucao_12_mais_alta)   
10          (Mae_30_34_alta)   
11   (Pre_natal_7_mais_alto)   
14  (Instrucao_12_mais_alta)   
..                       ...   
6         (Queda_nascimento)   
23        (Queda_nascimento)   
29        (Queda_nascimento)   
41        (Queda_nascimento)   
88        (Queda_nascimento)   

                                          consequents   support  confidence  \
8                            (Instrucao_12_mais_alta)  0.500000    1.000000   
9                                    (Mae_30_34_alta)  0.500000    1.000000   
10                            (Pre_natal_7_mais_alto)  0.500000    1.000000   
11                                   (Mae_30_34_alta)  0.500000    1.000000   
14                            (Pre_natal_7_mais_alto)  0.500000    1.000000   
..                                                ...       ...         ...   
6                                      (Cesario_alto)  0.42857

In [15]:
regras_queda = regras[
    regras['consequents'].astype(str).str.contains('Queda_nascimento') |
    regras['antecedents'].astype(str).str.contains('Queda_nascimento')
]

regras_queda = regras_queda.sort_values(
    by=['confidence', 'support'],
    ascending=False
)

print(regras_queda[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

                                   antecedents  \
20          (Queda_nascimento, Mae_30_34_alta)   
21  (Queda_nascimento, Instrucao_12_mais_alta)   
26          (Queda_nascimento, Mae_30_34_alta)   
27   (Queda_nascimento, Pre_natal_7_mais_alto)   
37  (Queda_nascimento, Instrucao_12_mais_alta)   
..                                         ...   
6                           (Queda_nascimento)   
23                          (Queda_nascimento)   
29                          (Queda_nascimento)   
41                          (Queda_nascimento)   
88                          (Queda_nascimento)   

                                          consequents   support  confidence  \
20                           (Instrucao_12_mais_alta)  0.428571    1.000000   
21                                   (Mae_30_34_alta)  0.428571    1.000000   
26                            (Pre_natal_7_mais_alto)  0.428571    1.000000   
27                                   (Mae_30_34_alta)  0.428571    1.000000   
37  

In [16]:
base_regras.to_csv('base_regras_associacao.csv', index=False)
base_binaria.to_csv('base_binaria_apriori.csv', index=False)
regras.to_csv('regras_associacao.csv', index=False)
regras_queda.to_csv('regras_queda_natalidade.csv', index=False)

print('Arquivos salvos com sucesso.')

Arquivos salvos com sucesso.
